### 01. Análisis Exploratorio de los Datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

In [30]:
data = pd.read_csv('./data/model_data.csv', thousands=',')

data.head(15)	

,AÑO,PIB a precios corrientes (miles de millones de pesos),Valor Agregado Pereira,Tasa de Desocupación (TD) Risaralda,Tasa de Desocupación (TD) Pereira ]A.M.,Exportaciones (dólares FOB),Importaciones (dólares CIF),Remesas (millones de dólares),Total empresas Risaralda,Inversión Neta en Sociedades jurisdicción CCP (millones de pesos),Población Risaralda,Población Pereira,Incidencia de Pobreza Multidimensional Risaralda,IDC Valor normalizado,Ranking IDC,Valor normalizado,Posición ranking
0,2005.0,5512.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,877974.0,426566.0,NaN,NaN,NaN,NaN,NaN
1,2006.0,6342.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,62914.0,884021.0,430198.0,NaN,NaN,NaN,NaN,NaN
2,2007.0,6852.0,NaN,11.7,13.3,NaN,NaN,NaN,NaN,96549.0,889400.0,433632.0,NaN,NaN,NaN,NaN,NaN
3,2008.0,7438.0,NaN,12.5,13.5,413200419.0,258538203.0,NaN,NaN,90152.0,893797.0,436494.0,NaN,NaN,NaN,NaN,NaN
4,2009.0,7969.0,NaN,17.8,20.0,474311821.0,214426608.0,NaN,NaN,65428.0,898126.0,439370.0,NaN,NaN,NaN,NaN,NaN
5,2010.0,8420.0,NaN,18.1,20.2,568427318.0,266809312.0,NaN,NaN,56130.0,902994.0,442493.0,NaN,NaN,NaN,NaN,NaN
6,2011.0,8980.0,4573.0,14.9,16.6,678286373.0,330730202.0,NaN,NaN,54660.0,907896.0,445438.0,NaN,NaN,NaN,NaN,NaN
7,2012.0,9629.0,5076.0,14.9,16.0,610562461.0,490691999.0,NaN,NaN,75415.0,912087.0,448174.0,NaN,NaN,NaN,NaN,NaN
8,2013.0,10613.0,5745.0,12.9,13.9,472667210.0,440431302.0,NaN,NaN,84258.0,915413.0,450422.0,NaN,NaN,NaN,NaN,NaN
9,2014.0,11728.0,6447.0,12.4,13.7,636030010.0,484273138.0,389.9,NaN,75593.0,918965.0,452676.0,NaN,NaN,NaN,NaN,NaN


In [31]:
desc_vars, control_vars = data.columns.tolist()[:12], data.columns.tolist()[12:]

print(f"Variables Descriptivas: {desc_vars}")
print(f"Variables de Control: {control_vars}")

Variables Descriptivas: ['AÑO', 'PIB a precios corrientes (miles de millones de pesos)', 'Valor Agregado Pereira', 'Tasa de Desocupación (TD) Risaralda', 'Tasa de Desocupación (TD) Pereira ]A.M.', 'Exportaciones (dólares FOB)', 'Importaciones (dólares CIF)', 'Remesas (millones de dólares)', 'Total empresas Risaralda', 'Inversión Neta en Sociedades jurisdicción CCP (millones de pesos)', 'Población Risaralda', 'Población Pereira']
Variables de Control: ['Incidencia de Pobreza Multidimensional Risaralda', 'IDC Valor normalizado', 'Ranking IDC', 'Valor normalizado', 'Posición ranking']


### 1.1 Generación de Variables Adicionales
Con el fin de preparar la información para la estimación de los modelos econométricos definidos, se procedió a construir una serie de variables transformadas, derivadas y estandarizadas que permiten mejorar la estabilidad estadística de los datos, facilitar la interpretación económica de los resultados y asegurar que las series cumplan los requisitos mínimos de estacionariedad y comparabilidad temporal.

Las transformaciones realizadas arrojan las siguientes variables esperadas:

##### Cálculo de Logaritmos Naturales (ln):
  - $ln(PIB)$ - PIB Real
  - $ln(EXP)$ - Exportaciones
  - $ln(IMP)$ - Importaciones
  - $ln(EMP)$ - Cantidad de Empresas
  - $ln(INV)$ - Inversión
  - $ln(REM)$ - Remeses
  - $ln(POB)$ - Población de Risaralda (Proxy Pereira)


In [32]:
cols_to_transform = data.columns.tolist()[1:12] # Nombre de las variables a transformar

# Calculo de Variables Descriptivas en Logaritmos Naturales (ln)
for col in cols_to_transform:
  ln_col = f'ln_{col}' # Nombre de la variable transformada

  if ln_col in data.columns:
    data.drop(columns=[ln_col], inplace=True)

  original_idx = data.columns.get_loc(col)

  data.insert(
    loc=original_idx + 1,
    column=ln_col,
    value=np.log(data[col]).round(4)
  )

# Calculo de Variables Representadas como Tasas de Cambio (Diferencias Logarítmicas)
for col in cols_to_transform:
  delta_col = f'delta_ln_{col}'

  if delta_col in data.columns:
    data.drop(columns=[delta_col], inplace=True)

  ln_col_idx = data.columns.get_loc(f'ln_{col}')

  data.insert(
    loc=ln_col_idx + 1,
    column=delta_col,
    value=data[f'ln_{col}'].diff().round(4)
  )


data.head(15)

,AÑO,PIB a precios corrientes (miles de millones de pesos),ln_PIB a precios corrientes (miles de millones de pesos),delta_ln_PIB a precios corrientes (miles de millones de pesos),Valor Agregado Pereira,ln_Valor Agregado Pereira,delta_ln_Valor Agregado Pereira,Tasa de Desocupación (TD) Risaralda,ln_Tasa de Desocupación (TD) Risaralda,delta_ln_Tasa de Desocupación (TD) Risaralda,...,ln_Población Risaralda,delta_ln_Población Risaralda,Población Pereira,ln_Población Pereira,delta_ln_Población Pereira,Incidencia de Pobreza Multidimensional Risaralda,IDC Valor normalizado,Ranking IDC,Valor normalizado,Posición ranking
0,2005.0,5512.0,8.6147,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,13.6854,NaN,426566.0,12.9635,NaN,NaN,NaN,NaN,NaN,NaN
1,2006.0,6342.0,8.7549,0.1402,NaN,NaN,NaN,NaN,NaN,NaN,...,13.6922,0.0068,430198.0,12.9720,0.0085,NaN,NaN,NaN,NaN,NaN
2,2007.0,6852.0,8.8323,0.0774,NaN,NaN,NaN,11.7,2.4596,NaN,...,13.6983,0.0061,433632.0,12.9800,0.0080,NaN,NaN,NaN,NaN,NaN
3,2008.0,7438.0,8.9144,0.0821,NaN,NaN,NaN,12.5,2.5257,0.0661,...,13.7032,0.0049,436494.0,12.9865,0.0065,NaN,NaN,NaN,NaN,NaN
4,2009.0,7969.0,8.9833,0.0689,NaN,NaN,NaN,17.8,2.8792,0.3535,...,13.7081,0.0049,439370.0,12.9931,0.0066,NaN,NaN,NaN,NaN,NaN
5,2010.0,8420.0,9.0384,0.0551,NaN,NaN,NaN,18.1,2.8959,0.0167,...,13.7135,0.0054,442493.0,13.0002,0.0071,NaN,NaN,NaN,NaN,NaN
6,2011.0,8980.0,9.1028,0.0644,4573.0,8.4279,NaN,14.9,2.7014,-0.1945,...,13.7189,0.0054,445438.0,13.0068,0.0066,NaN,NaN,NaN,NaN,NaN
7,2012.0,9629.0,9.1725,0.0697,5076.0,8.5323,0.1044,14.9,2.7014,0.0000,...,13.7235,0.0046,448174.0,13.0129,0.0061,NaN,NaN,NaN,NaN,NaN
8,2013.0,10613.0,9.2698,0.0973,5745.0,8.6561,0.1238,12.9,2.5572,-0.1442,...,13.7271,0.0036,450422.0,13.0179,0.0050,NaN,NaN,NaN,NaN,NaN
9,2014.0,11728.0,9.3697,0.0999,6447.0,8.7714,0.1153,12.4,2.5177,-0.0395,...,13.7310,0.0039,452676.0,13.0229,0.0050,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Exportar matrices de correlación
import os

# Crear directorio de salida si no existe
output_dir = './output'
os.makedirs(output_dir, exist_ok=True)

# Exportar correlaciones
corr_original.to_csv(f'{output_dir}/correlation_original_vars.csv')
corr_ln.to_csv(f'{output_dir}/correlation_log_vars.csv')
corr_delta.to_csv(f'{output_dir}/correlation_change_rates.csv')

# Exportar top correlaciones
corr_df.to_csv(f'{output_dir}/top_correlations_original.csv', index=False)
corr_df_ln.to_csv(f'{output_dir}/top_correlations_log.csv', index=False)
corr_df_delta.to_csv(f'{output_dir}/top_correlations_change_rates.csv', index=False)

print("✓ Matrices de correlación exportadas exitosamente:")
print(f"  - {output_dir}/correlation_original_vars.csv")
print(f"  - {output_dir}/correlation_log_vars.csv")
print(f"  - {output_dir}/correlation_change_rates.csv")
print(f"\n✓ Top correlaciones exportadas:")
print(f"  - {output_dir}/top_correlations_original.csv")
print(f"  - {output_dir}/top_correlations_log.csv")
print(f"  - {output_dir}/top_correlations_change_rates.csv")